# 2D-to-3D Game Models Pipeline — Remote Test

**This notebook is for testing only.** The actual pipeline runs natively via `python run.py`.

Requirements: Colab with GPU runtime (Runtime → Change runtime type → T4 GPU)

**IMPORTANT: Do NOT use "Run All". Run cells one by one** — cell 3 will ask you to upload an image from your phone.

## 0. Check GPU

In [ ]:
!nvidia-smi
import torch
print(f"\nCUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

## 1. Clone repo & install dependencies

In [ ]:
import os
os.chdir('/content')

# Clone the repo
!rm -rf 2d-to-3d-game-models
!git clone -b claude/image-to-3d-pipeline-CnSII https://github.com/pmikola/2d-to-3d-game-models.git
os.chdir('2d-to-3d-game-models')
!git log --oneline -5

In [ ]:
# Install core dependencies (skip torch — Colab already has it)
# Install onnxruntime-gpu FIRST to avoid rembg crash with CPU-only onnxruntime
!pip install -q onnxruntime-gpu
!pip install -q trimesh pygltflib xatlas Pillow "rembg[gpu]" numpy \
    diffusers transformers accelerate safetensors \
    pyyaml tqdm huggingface_hub

# Verify rembg imports cleanly
try:
    from rembg import remove
    print("✅ rembg installed successfully")
except Exception as e:
    print(f"⚠️ rembg issue: {e}")
    print("Background removal will be skipped — pipeline still works without it.")

## 2. Test: Device Detection

In [ ]:
from pipeline.device import detect_device, log_device_info
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(name)s: %(message)s')

config = detect_device()
log_device_info(config)
print(f"\nDevice: {config.device}")
print(f"Dtype: {config.dtype}")
print(f"Viewpoints: {config.texture_num_viewpoints}")
print(f"DDIM steps: {config.texture_ddim_steps}")
print("\n✅ Device detection works!")

## 3. Upload Your Image (from phone/PC)

Tap "Choose Files" to upload a PNG or JPG from your device.

In [ ]:
from google.colab import files
from PIL import Image
import numpy as np

print("📱 Upload your image (PNG/JPG):")
uploaded = files.upload()

if uploaded:
    # Use the first uploaded file
    filename = list(uploaded.keys())[0]
    import shutil
    shutil.copy(filename, '/content/test_input.png')
    img = Image.open('/content/test_input.png')
    print(f"\nUploaded: {filename} ({img.size[0]}x{img.size[1]}, {img.mode})")
    display(img.resize((300, 300)))
else:
    # Fallback: create synthetic test image
    print("No file uploaded — using synthetic test image.")
    test_img = np.zeros((400, 400, 3), dtype=np.uint8)
    test_img[:] = np.random.randint(100, 200, (400, 400, 3), dtype=np.uint8)
    y, x = np.ogrid[-200:200, -200:200]
    mask = x**2 + y**2 < 120**2
    test_img[mask] = [220, 140, 60]
    mask_inner = x**2 + y**2 < 40**2
    test_img[mask_inner] = [255, 200, 100]
    test_image = Image.fromarray(test_img)
    test_image.save('/content/test_input.png')
    display(test_image.resize((200, 200)))

In [ ]:
from pipeline.preprocess import preprocess_image

# Try with background removal; fall back to without if rembg fails
try:
    preprocessed = preprocess_image(
        '/content/test_input.png',
        target_size=512,
        remove_bg=True,
        use_gpu=False,
    )
    print("✅ Preprocessing with background removal succeeded!")
except SystemExit:
    print("⚠️ rembg crashed (onnxruntime issue). Retrying without background removal...")
    preprocessed = preprocess_image(
        '/content/test_input.png',
        target_size=512,
        remove_bg=False,
        use_gpu=False,
    )
    print("✅ Preprocessing without background removal succeeded!")

print(f"Result: {preprocessed.size}, mode={preprocessed.mode}")
display(preprocessed.resize((200, 200)))

## 4. Test: Mesh Repair

In [ ]:
import trimesh
from pipeline.mesh_repair import validate_mesh, repair_and_prepare

# Create a test mesh (icosphere with some issues)
mesh = trimesh.creation.icosphere(subdivisions=3, radius=1.0)
print(f"Original mesh: {len(mesh.vertices)} verts, {len(mesh.faces)} faces")

# Validate
metrics = validate_mesh(mesh)
print(f"Metrics: {metrics}")

# Run full repair pipeline
repaired = repair_and_prepare(mesh, smooth_iterations=3)
print(f"\nRepaired mesh: {len(repaired.vertices)} verts, {len(repaired.faces)} faces")
print("\n✅ Mesh repair works!")

In [ ]:
# Test decimation
from pipeline.mesh_repair import decimate_mesh

dense_mesh = trimesh.creation.icosphere(subdivisions=5, radius=1.0)
print(f"Dense mesh: {len(dense_mesh.faces)} faces")

decimated = decimate_mesh(dense_mesh, target_ratio=0.25)
print(f"Decimated mesh: {len(decimated.faces)} faces")
print(f"Reduction: {100*(1 - len(decimated.faces)/len(dense_mesh.faces)):.1f}%")
print("\n✅ Mesh decimation works!")

## 5. Test: UV Unwrapping (xatlas)

In [ ]:
from pipeline.geometry import unwrap_uvs, normalize_mesh, save_mesh_as_obj

mesh = trimesh.creation.icosphere(subdivisions=3, radius=1.0)
normalize_mesh(mesh)

print(f"Before UV unwrap: {len(mesh.vertices)} verts")
unwrap_uvs(mesh)
print(f"After UV unwrap: {len(mesh.vertices)} verts")

# Check UVs exist
has_uvs = hasattr(mesh.visual, 'uv') and mesh.visual.uv is not None
print(f"Has UVs: {has_uvs}")
if has_uvs:
    print(f"UV shape: {mesh.visual.uv.shape}")

# Save as OBJ
obj_path = save_mesh_as_obj(mesh, '/content/test_mesh')
print(f"Saved OBJ: {obj_path}")
print("\n✅ UV unwrapping works!")

## 6. Test: PBR Map Generation

In [ ]:
from pipeline.pbr_maps import generate_pbr_maps, save_pbr_maps

# Use the test image as a fake albedo texture
albedo = Image.open('/content/test_input.png').resize((512, 512))

pbr_maps = generate_pbr_maps(albedo, strength=1.5)

print("Generated PBR maps:")
for name, img in pbr_maps.items():
    print(f"  {name}: {img.size}, mode={img.mode}")

# Display them
from IPython.display import display
print("\nAlbedo → Normal → Roughness → Metallic:")
row = Image.new('RGB', (512*4, 512))
row.paste(albedo, (0, 0))
row.paste(pbr_maps['normal'], (512, 0))
row.paste(pbr_maps['roughness'].convert('RGB'), (1024, 0))
row.paste(pbr_maps['metallic'].convert('RGB'), (1536, 0))
display(row.resize((800, 200)))

# Save
saved = save_pbr_maps(pbr_maps, '/content/test_pbr')
print(f"\nSaved: {saved}")
print("\n✅ PBR map generation works!")

## 7. Test: GLB Export with PBR

In [ ]:
from pipeline.export import export_to_glb, validate_glb

# Export mesh + texture + PBR maps to GLB
glb_path = '/content/test_output.glb'
export_to_glb(
    mesh_path='/content/test_mesh/mesh.obj',
    texture_path='/content/test_input.png',
    output_path=glb_path,
    normal_map_path='/content/test_pbr/normal_map.png',
    roughness_map_path='/content/test_pbr/roughness_map.png',
    metallic_map_path='/content/test_pbr/metallic_map.png',
)

# Validate
info = validate_glb(glb_path)
print(f"\nGLB validation: {info}")

import os
size_mb = os.path.getsize(glb_path) / (1024*1024)
print(f"GLB file size: {size_mb:.2f} MB")
print("\n✅ GLB export with PBR textures works!")

## 8. Test: YAML Config Loading

In [ ]:
import sys
sys.argv = ['run.py', '--input', 'dummy.png']  # Fake argv for argparse

from run import load_config_from_yaml

config = load_config_from_yaml('configs/default.yaml')
print("Loaded config from YAML:")
print(f"  backend: {config.backend}")
print(f"  target_size: {config.target_size}")
print(f"  remove_background: {config.remove_background}")
print(f"  geometry_seed: {config.geometry_seed}")
print(f"  geometry_steps: {config.geometry_steps}")
print(f"  mesh_repair: {config.mesh_repair}")
print(f"  mesh_smooth_iterations: {config.mesh_smooth_iterations}")
print(f"  generate_pbr: {config.generate_pbr}")
print(f"  force_cpu: {config.force_cpu}")
print("\n✅ YAML config loading works!")

## 9. Test: Full Pipeline (Geometry → Repair → UV → Texture → PBR → GLB)

This tests the complete pipeline **without** Hi3DGen/Hunyuan3D models (which require separate installation).  
Instead, we inject a synthetic mesh to test everything from mesh repair onwards.

In [ ]:
import trimesh
import tempfile
import time
from pathlib import Path
from pipeline.mesh_repair import repair_and_prepare
from pipeline.geometry import normalize_mesh, unwrap_uvs, save_mesh_as_obj
from pipeline.pbr_maps import generate_pbr_maps, save_pbr_maps
from pipeline.export import export_textured_dir_to_glb, validate_glb
from pipeline.preprocess import preprocess_image

start = time.time()

# Create output dir first
Path('/content/pipeline_test').mkdir(exist_ok=True)

# Stage 0: Preprocess YOUR uploaded image
print("[0/5] Preprocessing your image...")
try:
    preprocessed = preprocess_image('/content/test_input.png', target_size=512, remove_bg=True)
except (SystemExit, Exception):
    preprocessed = preprocess_image('/content/test_input.png', target_size=512, remove_bg=False)
preprocessed.save('/content/pipeline_test/preprocessed.png')
display(preprocessed.resize((200, 200)))

# Stage 1: Simulate geometry generation (synthetic mesh — real models need Hi3DGen/Hunyuan3D)
print("[1/5] Simulating geometry generation (synthetic mesh)...")
mesh = trimesh.creation.icosphere(subdivisions=4, radius=1.0)
mesh.vertices += np.random.normal(0, 0.01, mesh.vertices.shape)
print(f"  Generated: {len(mesh.vertices)} verts, {len(mesh.faces)} faces")

# Stage 2: Mesh repair
print("[2/5] Repairing mesh...")
mesh = repair_and_prepare(mesh, smooth_iterations=3)
print(f"  Repaired: {len(mesh.vertices)} verts, {len(mesh.faces)} faces")

# Stage 3: Normalize + UV unwrap
print("[3/5] Normalizing + UV unwrapping...")
normalize_mesh(mesh)
unwrap_uvs(mesh)

with tempfile.TemporaryDirectory() as tmp_dir:
    obj_path = save_mesh_as_obj(mesh, tmp_dir)
    
    # Stage 4: Create texture + PBR maps from your image
    print("[4/5] Creating texture + PBR maps from your image...")
    textured_dir = Path(tmp_dir) / 'textured'
    textured_dir.mkdir()
    preprocessed.save(str(textured_dir / 'texture_atlas.png'))
    
    import shutil
    shutil.copy(obj_path, str(textured_dir / 'mesh_textured.obj'))
    
    pbr_maps = generate_pbr_maps(preprocessed, strength=1.5)
    save_pbr_maps(pbr_maps, str(textured_dir / 'pbr'))
    
    # Show PBR maps
    print("  PBR maps generated:")
    row = Image.new('RGB', (512*4, 512))
    row.paste(preprocessed.resize((512,512)), (0, 0))
    row.paste(pbr_maps['normal'].resize((512,512)), (512, 0))
    row.paste(pbr_maps['roughness'].convert('RGB').resize((512,512)), (1024, 0))
    row.paste(pbr_maps['metallic'].convert('RGB').resize((512,512)), (1536, 0))
    display(row.resize((800, 200)))
    
    # Stage 5: Export to GLB
    print("[5/5] Exporting to GLB...")
    output_glb = '/content/pipeline_test/full_test.glb'
    export_textured_dir_to_glb(str(textured_dir), output_glb)

elapsed = time.time() - start

info = validate_glb(output_glb)
size_mb = os.path.getsize(output_glb) / (1024*1024)

print(f"\n{'='*50}")
print(f"FULL PIPELINE TEST COMPLETE")
print(f"  Output: {output_glb}")
print(f"  GLB size: {size_mb:.2f} MB")
print(f"  Validation: {info}")
print(f"  Duration: {elapsed:.1f}s")
print(f"{'='*50}")

# Auto-download the GLB to your phone
from google.colab import files
print("\n📱 Downloading GLB to your device...")
files.download(output_glb)
print("✅ Done! Open the .glb file in any 3D viewer.")

## 10. (Optional) Test with Hi3DGen Model

Uncomment and run if you want to test the actual geometry generation.  
This downloads ~5GB of model weights and takes ~5 min on a T4.

In [ ]:
# # Install Hi3DGen
# !git clone https://github.com/bytedance/Hi3DGen.git /content/Hi3DGen
# !cd /content/Hi3DGen && pip install -q -r requirements.txt
# 
# # Run full pipeline with Hi3DGen
# os.chdir('/content/2d-to-3d-game-models')
# !python run.py --input /content/test_input.png \
#     --output /content/hi3dgen_test.glb \
#     --hi3dgen-path /content/Hi3DGen \
#     --verbose

## 11. (Optional) Test with Hunyuan3D-2.1

Uncomment to test the single-stage Hunyuan3D backend.  
Downloads ~10GB of model weights.

In [ ]:
# # Install Hunyuan3D
# !pip install -q hy3dgen
# 
# # Run full pipeline with Hunyuan3D
# os.chdir('/content/2d-to-3d-game-models')
# !python run.py --input /content/test_input.png \
#     --output /content/hunyuan3d_test.glb \
#     --backend hunyuan3d \
#     --verbose

## 12. Download Results to Your Phone

Tap the download link to save the .GLB file to your device.
You can view it in any 3D viewer app or upload to https://gltf-viewer.donmccurdy.com/

In [ ]:
from google.colab import files
import glob
import os

glb_files = sorted(glob.glob('/content/**/*.glb', recursive=True))
print(f"📦 GLB files generated: {len(glb_files)}")
for f in glb_files:
    size = os.path.getsize(f) / (1024*1024)
    print(f"  {f} ({size:.2f} MB)")

# Download all GLB files
print("\n📱 Downloading to your device...")
for f in glb_files:
    print(f"  ⬇️  {os.path.basename(f)}")
    files.download(f)

# Also download PBR maps if they exist
pbr_files = glob.glob('/content/test_pbr/*.png') + glob.glob('/content/pipeline_test/**/*.png', recursive=True)
if pbr_files:
    print(f"\n🎨 PBR maps available ({len(pbr_files)} files):")
    for f in pbr_files:
        print(f"  {os.path.basename(f)}")
    # Uncomment to download PBR maps too:
    # for f in pbr_files:
    #     files.download(f)

---

### Test Summary

| Test | What it validates |
|------|------------------|
| Device detection | GPU/CPU detection, VRAM checking, dtype selection |
| Preprocessing | rembg background removal, resize, quality checks |
| Mesh repair | Topology fix, smoothing, decimation, validation |
| UV unwrapping | xatlas UV generation, OBJ export |
| PBR maps | Normal, roughness, metallic map generation |
| GLB export | Embedded textures + PBR materials, validation |
| YAML config | Config file loading + field mapping |
| Full pipeline | End-to-end integration (synthetic mesh) |
| Hi3DGen (opt) | Real geometry generation on T4 GPU |
| Hunyuan3D (opt) | Single-stage geometry + PBR texturing |